### Stage 2: Data Transformation
Tokenizes the ingested dataset and saves transformed splits to disk

In [30]:
import os
%pwd

'c:\\Users'

In [31]:
os.chdir("../")
%pwd

'c:\\'

In [32]:
def read_yaml(path):
    with open(path, "r") as f:
        return ConfigBox(yaml.safe_load(f))

In [37]:
import subprocess
result = subprocess.run(
    ['where', '/r', r'C:\Users\HP\Text-Summarizer', 'config.yaml'],
    capture_output=True, text=True
)
print(result.stdout)

C:\Users\HP\Text-Summarizer\config\config.yaml



In [39]:
import os

os.chdir(r"C:\Users\HP\Text-Summarizer")  # absolute path, no ambiguity
print(os.getcwd())
print(os.path.exists("config/config.yaml"))  # must print True

C:\Users\HP\Text-Summarizer
True


In [38]:
import os

for root, dirs, files in os.walk(r'C:\Users\HP\Text-Summarizer'):
    # skip venv and hidden folders
    dirs[:] = [d for d in dirs if d not in ['venv', '.git', '__pycache__', 'artifacts']]
    level = root.replace(r'C:\Users\HP\Text-Summarizer', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        print(f'{indent}  {file}')

Text-Summarizer/
  .gitignore
  app.py
  Dockerfile
  main.py
  params.yaml
  README.md
  requirements.txt
  setup.py
  template.py
  .github/
    workflows/
      .gitkeep
  config/
    config.yaml
  ingested_test/
    test.csv
  ingested_train/
    train.csv
  logs/
    continuous_logs.log
  raw_data/
  research/
    data_ingestion.ipynb
    data_transformation.ipynb
    research.ipynb
    textsummarizer.ipynb
  src/
    textsummarizer/
      __init__.py
      components/
        data_ingestion.py
        __init__.py
      config/
        configuration.py
        __init__.py
      constants/
        __init__.py
      entity/
        __init__.py
      logging/
        __init__.py
      pipeline/
        stage_1_data_ingestion_pipeline.py
        __init__.py
      utils/
        common.py
        __init__.py


### 1. Config Entity

In [40]:
from dataclasses import dataclass
from pathlib import Path
from box import ConfigBox

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    ingested_train_path: Path
    ingested_test_path: Path
    tokenizer_name: str
    input_column: str
    target_column: str
    max_input_length: int
    max_target_length: int

In [41]:
import yaml
from pathlib import Path
from box import ConfigBox

def read_yaml(path):
    with open(path, "r") as f:
        return ConfigBox(yaml.safe_load(f))

# Use this local version instead of src.utils
config = read_yaml(Path("config/config.yaml"))
params = read_yaml(Path("params.yaml"))

print(config.artifacts.data_transformation.tokenizer_name)
# Should print: google/pegasus-xsum

google/pegasus-xsum


### 2. Configuration Manager

In [42]:
from src.textsummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.textsummarizer.utils.common import read_yaml, create_directories

CONFIG_FILE_PATH = CONFIG_FILE_PATH
PARAMS_FILE_PATH = PARAMS_FILE_PATH

class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(Path("config/config.yaml"))  # local read_yaml above
        self.params = read_yaml(Path("params.yaml"))

    def get_data_transformation_config(self) -> DataTransformationConfig:
        cfg    = self.config.artifacts.data_transformation
        params = self.params.data_transformation
        ing    = self.config.artifacts.data_ingestion
        os.makedirs(cfg.root_dir, exist_ok=True)
        return DataTransformationConfig(
            root_dir=Path(cfg.root_dir),
            ingested_train_dir=Path(ing.ingested_train_dir),
            ingested_test_dir=Path(ing.ingested_test_dir),
            tokenizer_name=cfg.tokenizer_name,
            input_column=params.input_column,
            target_column=params.target_column,
            max_input_length=params.max_input_length,
            max_target_length=params.max_target_length,
        )
print("ConfigurationManager and DataTransformationConfig are set up correctly.")

ConfigurationManager and DataTransformationConfig are set up correctly.


### Data Transformation Component

In [43]:
from datasets import load_from_disk
from transformers import AutoTokenizer
from src.textsummarizer.logging import logger

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def tokenize_batch(self, batch):
        input_encodings = self.tokenizer(
            batch[self.config.input_column],
            max_length=self.config.max_input_length,
            truncation=True,
            padding="max_length",
        )

        # Use text_target for newer tokenozers (replaces as target_tokenizer)
        target_encodings = self.tokenizer(
            text_target=batch[self.config.target_column],
            max_length=self.config.max_target_length,
            truncation=True,
            padding="max_length",
        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids'],
        }

    def transform(self):
        train_ds = load_from_disk(str(self.config.ingested_train_dir))
        test_ds = load_from_disk(str(self.config.ingested_test_dir))

        print(f"Train size before: {len(train_ds)}")
        print(f"Test size before: {len(test_ds)}")

        train_ds = train_ds.map(self.tokenize_batch, batched=True)
        test_ds = test_ds.map(self.tokenize_batch, batched=True)

        train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
        test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

        train_ds.save_to_disk(str(self.config.root_dir / "transformed_train"))
        test_ds.save_to_disk(str(self.config.root_dir / "transformed_test"))

        print('Transformation complete. Splits savved to disk:', self.config.root_dir)
        return train_ds, test_ds

### Run the Pipeline

In [ ]:
try:
    config_manager = ConfigurationManager()
    transform_config = config_manager.get_data_transformation_config()
    transformation = DataTransformation(config=transform_config)
    train_ds, test_ds = transformation.transform()
except Exception as e:
    raise e